In [ ]:
import numpy as np
from typing import List


class Hungarian:
    N: int  # the dimension of the profit matrix
    tmp_tensor: np.ndarray

    def __init__(self, profit_matrix):
        self.profit_matrix = profit_matrix
        self.cost_matrix = self.create_cost_matrix(profit_matrix)
        self.step1()
        self.ans_pos = self.step_2_3()
        self.ans, self.ans_mat = self.ans_calculation(profit_matrix, self.ans_pos)

    def create_cost_matrix(self, profit_matrix):
        self.N = profit_matrix.shape[0]
        max_value = np.max(profit_matrix)
        return max_value - profit_matrix

    def min_zero_row(self, zero_mat, mark_zero):
        """
        The function can be splitted into two steps:
        #1 The function is used to find the row which containing the fewest 0.
        #2 Select the zero number on the row, and then marked the element corresponding row and column as False
        """

        # Find the row
        min_row = [99999, -1]

        for row_num in range(zero_mat.shape[0]):
            if np.sum(zero_mat[row_num] == True) > 0 and min_row[0] > np.sum(
                zero_mat[row_num] == True
            ):
                min_row = [np.sum(zero_mat[row_num] == True), row_num]

        # Marked the specific row and column as False
        zero_index = np.where(zero_mat[min_row[1]] == True)[0][0]
        mark_zero.append((min_row[1], zero_index))
        zero_mat[min_row[1], :] = False
        zero_mat[:, zero_index] = False

    def mark_matrix(self, mat):
        """
        Finding the returning possible solutions for LAP problem.
        """

        # Transform the matrix to boolean matrix(0 = True, others = False)
        cur_mat = mat
        zero_bool_mat = cur_mat == 0
        zero_bool_mat_copy = zero_bool_mat.copy()

        # Recording possible answer positions by marked_zero
        marked_zero = []
        while True in zero_bool_mat_copy:
            self.min_zero_row(zero_bool_mat_copy, marked_zero)

        # Recording the row and column positions seperately.
        marked_zero_row = []
        marked_zero_col = []
        for i in range(len(marked_zero)):
            marked_zero_row.append(marked_zero[i][0])
            marked_zero_col.append(marked_zero[i][1])

        # Step 2-2-1
        non_marked_row = list(set(range(cur_mat.shape[0])) - set(marked_zero_row))

        marked_cols = []
        check_switch = True
        while check_switch:
            check_switch = False
            for i in range(len(non_marked_row)):
                row_array = zero_bool_mat[non_marked_row[i], :]
                for j in range(row_array.shape[0]):
                    # Step 2-2-2
                    if row_array[j] == True and j not in marked_cols:
                        # Step 2-2-3
                        marked_cols.append(j)
                        check_switch = True

            for row_num, col_num in marked_zero:
                # Step 2-2-4
                if row_num not in non_marked_row and col_num in marked_cols:
                    # Step 2-2-5
                    non_marked_row.append(row_num)
                    check_switch = True
        # Step 2-2-6
        marked_rows = list(set(range(mat.shape[0])) - set(non_marked_row))

        return (marked_zero, marked_rows, marked_cols)

    def step1(self):
        self.tmp_tensor = self.cost_matrix.copy()
        for i in range(self.N):
            min_value = min(self.tmp_tensor[i])
            for j in range(self.N):
                self.tmp_tensor[i][j] -= min_value
        for j in range(self.N):
            col_values = [self.tmp_tensor[i][j] for i in range(self.N)]
            min_value = min(col_values)
            for i in range(self.N):
                self.tmp_tensor[i][j] -= min_value

    def step_2_3(self):
        tmp_tensor = self.tmp_tensor
        num_zero: int = 0
        while num_zero < self.N:
            pos, marked_rows, marked_cols = self.mark_matrix(np.array(tmp_tensor))
            num_zero = len(marked_rows) + len(marked_cols)
            if num_zero < self.N:
                tmp_tensor = self.adjust_matrix(tmp_tensor, marked_rows, marked_cols)
        return pos

    def adjust_matrix(self, mat, cover_rows, cover_cols):
        cur_mat = mat
        non_zero_element = []

        # Step 4-1
        for row in range(len(cur_mat)):
            if row not in cover_rows:
                for i in range(len(cur_mat[row])):
                    if i not in cover_cols:
                        non_zero_element.append(cur_mat[row][i])
        min_num = min(non_zero_element)

        # Step 4-2
        for row in range(len(cur_mat)):
            if row not in cover_rows:
                for i in range(len(cur_mat[row])):
                    if i not in cover_cols:
                        cur_mat[row, i] = cur_mat[row, i] - min_num
        # Step 4-3
        for row in range(len(cover_rows)):
            for col in range(len(cover_cols)):
                cur_mat[cover_rows[row], cover_cols[col]] = (
                    cur_mat[cover_rows[row], cover_cols[col]] + min_num
                )
        return cur_mat

    def ans_calculation(self, mat, pos):
        total = 0
        ans_mat = np.zeros((mat.shape[0], mat.shape[1]))
        for i in range(len(pos)):
            total += mat[pos[i][0], pos[i][1]]
            ans_mat[pos[i][0], pos[i][1]] = mat[pos[i][0], pos[i][1]]
        return total, ans_mat

    def get_answer(self):
        return self.ans, self.ans_mat


def main():

    profit_matrix = np.array([[7, 8, 5, 6], [6, 5, 9, 10], [4, 1, 10, 7], [3, 4, 6, 5]])
    solver = Hungarian(profit_matrix.copy())
    ans, ans_mat = solver.get_answer()
    print(f"Linear Assignment problem result: {ans:.0f}\n{ans_mat}")


if __name__ == "__main__":
    main()

Linear Assignment problem result: 31
[[ 7.  0.  0.  0.]
 [ 0.  0.  0. 10.]
 [ 0.  0. 10.  0.]
 [ 0.  4.  0.  0.]]


In [1]:
import numpy as np

cost_matrix = np.random.randint(10, size=(5, 5))
print(f"The cost matrix is:\n", cost_matrix)

The cost matrix is:
 [[7 6 3 2 9]
 [2 8 5 8 7]
 [4 1 4 1 7]
 [4 0 6 7 3]
 [0 6 6 9 4]]


ステップ 1：各列および各行から、その中の最小値を引く
まず、すべての列とすべての行について、それぞれの内部にある最小値を引きます。
最小値を引いた後、コスト行列は次のようになります。

In [ ]:
def step1(cost_matrix):
    """
    Step 1:すべての列とすべての行について、それぞれの内部にある最小値を引く
    """
    tmp_tesor = cost_matrix.copy()
    for i in range(cost_matrix.shape[0]):
        min_value = min(tmp_tesor[i])
        for j in range(cost_matrix.shape[1]):
            tmp_tesor[i][j] -= min_value
    for j in range(cost_matrix.shape[1]):
        col_values = [tmp_tesor[i][j] for i in range(cost_matrix.shape[0])]
        min_value = min(col_values)
        for i in range(cost_matrix.shape[0]):
            tmp_tesor[i][j] -= min_value
    return tmp_tesor

In [17]:
def step2(cost_matrix):
    """
    Step 2:0をカバーする最小の行と列の組み合わせを見つける
    """
    tmp_tensor = cost_matrix
    zero_bool_mat = (tmp_tensor == 0)
    return zero_bool_mat

In [18]:
def main():

    # The matrix who you want to find the maximum sum
    cost_matrix = np.array(
        [
            [7, 6, 2, 9, 2],
            [6, 2, 1, 3, 9],
            [5, 6, 8, 9, 5],
            [6, 8, 5, 8, 6],
            [9, 5, 6, 4, 7],
        ]
    )
    ans_pos = step1(cost_matrix.copy())
    ans_pos = step2(ans_pos)

    print(f"The result matrix is:\n", ans_pos)


if __name__ == "__main__":
    main()

The result matrix is:
 [[False False  True False  True]
 [False  True  True False False]
 [ True  True False False  True]
 [False False  True False False]
 [False  True False  True False]]


In [25]:
import numpy as np


class Hungarian:
    def __init__(self, cost_matrix: np.ndarray):
        # 元のコスト行列（答えの計算用に保持）
        self.original_cost = cost_matrix.astype(float)
        # 計算用にコピー
        self.cur_mat = self.original_cost.copy()
        # 次元数（正方行列を前提）
        self.dim = self.cur_mat.shape[0]

    def min_zero_row(self, zero_mat, mark_zero):
        """
        zero_mat: True/False のブール行列（True: 0 がある）
        mark_zero: 選んだ 0 の位置(行,列)を溜めるリスト
        """
        # 0(=True) の個数が最も少ない行を探す
        min_row = [99999, -1]  # [その行の0の個数, 行番号]

        for row_num in range(zero_mat.shape[0]):
            zero_count = np.sum(zero_mat[row_num] == True)
            if zero_count > 0 and min_row[0] > zero_count:
                min_row = [zero_count, row_num]

        # 見つけた行の中で最初の 0(=True) の列インデックスを取得
        zero_index = np.where(zero_mat[min_row[1]] == True)[0][0]

        # 選んだ 0 の位置を記録
        mark_zero.append((min_row[1], zero_index))

        # 同じ行・列の 0 は、今後使わないよう False にする
        zero_mat[min_row[1], :] = False
        zero_mat[:, zero_index] = False

    def step1(self):
        """
        Step 1:
        各行・各列から、その行・列内の最小値を引いて 0 を必ず含むようにする。
        """
        # 行ごとに最小値を引く
        for row_num in range(self.cur_mat.shape[0]):
            self.cur_mat[row_num] -= np.min(self.cur_mat[row_num])

        # 列ごとに最小値を引く
        for col_num in range(self.cur_mat.shape[1]):
            self.cur_mat[:, col_num] -= np.min(self.cur_mat[:, col_num])

    def step2(self):
        """
        Step 2:
        0 の位置をもとに「独立な 0」を選びつつ、
        それに基づいて行・列のカバー候補を求める。
        （実装は _mark_matrix に委譲）
        """
        # 0 かどうかのブール行列を作る
        cur_mat = self.cur_mat
        zero_bool_mat = cur_mat == 0
        zero_bool_mat_copy = zero_bool_mat.copy()

        # 選択した 0 の位置を記録
        marked_zero = []
        while True in zero_bool_mat_copy:
            self.min_zero_row(zero_bool_mat_copy, marked_zero)

        # 行と列を別々に記録
        marked_zero_row = []
        marked_zero_col = []
        for r, c in marked_zero:
            marked_zero_row.append(r)
            marked_zero_col.append(c)

        # Step 2-2-1: マークされていない行を求める
        non_marked_row = list(set(range(cur_mat.shape[0])) - set(marked_zero_row))

        marked_cols = []
        check_switch = True
        while check_switch:
            check_switch = False
            # まだマークされていない行を走査
            for r in non_marked_row:
                row_array = zero_bool_mat[r, :]
                for j in range(row_array.shape[0]):
                    if row_array[j] and j not in marked_cols:
                        marked_cols.append(j)
                        check_switch = True

            for row_num, col_num in marked_zero:
                if row_num not in non_marked_row and col_num in marked_cols:
                    non_marked_row.append(row_num)
                    check_switch = True

        marked_rows = list(set(range(cur_mat.shape[0])) - set(non_marked_row))

        return marked_zero, marked_rows, marked_cols

    def step3(self, marked_rows, marked_cols):
        """
        Step 3:
        マークされた行数 + 列数を数え、dim (次元) と比較する。
        戻り値: zero_count（= カバー線の本数）
        """
        zero_count = len(marked_rows) + len(marked_cols)
        return zero_count

    def step4(self, marked_rows, marked_cols):
        """
        Step 4:
        カバーされていない要素を用いて行列を調整し、新しい 0 を作る。
        （実装は _adjust_matrix に委譲）
        """
        cur_mat = self.cur_mat
        non_zero_element = []

        # Step 4-1: カバーされていない要素の最小値を探す
        for row in range(len(cur_mat)):
            if row not in marked_rows:
                for col in range(len(cur_mat[row])):
                    if col not in marked_cols:
                        non_zero_element.append(cur_mat[row][col])
        min_num = min(non_zero_element)

        # Step 4-2: カバーされていない要素から最小値を引く
        for row in range(len(cur_mat)):
            if row not in marked_rows:
                for col in range(len(cur_mat[row])):
                    if col not in marked_cols:
                        cur_mat[row, col] = cur_mat[row, col] - min_num

        # Step 4-3: 行・列両方カバーされている交点に最小値を足す
        for r in marked_rows:
            for c in marked_cols:
                cur_mat[r, c] = cur_mat[r, c] + min_num

        return cur_mat

    def solve(self):
        """
        ハンガリアン法本体。
        step1〜step4 を用いて、最終的な割当位置(行,列)のリストを返す。
        """
        # Step 1: 前処理
        self.step1()

        zero_count = 0
        ans_pos = None

        # 行列の次元数分のカバー線が得られるまで繰り返す
        while zero_count < self.dim:
            # Step 2: 0 のマーキングと行・列のカバー候補を取得
            ans_pos, marked_rows, marked_cols = self.step2()
            # Step 3: カバー線の本数を確認
            zero_count = self.step3(marked_rows, marked_cols)

            # まだ十分でなければ Step 4 で行列調整
            if zero_count < self.dim:
                self.step4(marked_rows, marked_cols)

        # 最終的な割当位置を返す
        return ans_pos


def ans_calculation(mat, pos):
    total = 0
    ans_mat = np.zeros((mat.shape[0], mat.shape[1]))
    for r, c in pos:
        total += mat[r, c]
        ans_mat[r, c] = mat[r, c]
    return total, ans_mat


def main():
    """Hungarian Algorithm demo"""

    cost_matrix = np.array(
        [
            [7, 6, 2, 9, 2],
            [6, 2, 1, 3, 9],
            [5, 6, 8, 9, 5],
            [6, 8, 5, 8, 6],
            [9, 5, 6, 4, 7],
        ],
        dtype=float,
    )

    # 最小コスト割当
    hungarian_min = Hungarian(cost_matrix.copy())
    ans_pos = hungarian_min.solve()
    ans, ans_mat = ans_calculation(cost_matrix, ans_pos)
    print("=== Min assignment ===")
    print(f"Result: {ans:.0f}\n{ans_mat}")


if __name__ == "__main__":
    main()

IndexError: index 0 is out of bounds for axis 0 with size 0

In [80]:
import numpy as np

INF_ = 1e9
class Hungarian:
    def __init__(self, cost_matrix: np.ndarray):
        self.origin_tensor = cost_matrix.copy()
        self.now_tensor = cost_matrix.copy()
        self.N = cost_matrix.shape[0]
        self.now_tensor = self.step1(self.now_tensor)
        while True:
            self.assignment = self.step2(self.now_tensor)
            if self.assignment is not None:
                break
            marked_tensor = self.step3(self.now_tensor)
            print("After step3 processing, the marked tensor is:")
            print(f"{marked_tensor}")
            final_tensor = self.step4(self.now_tensor, marked_tensor)
            print(f"{final_tensor}")
            payment = 0
        for assign in self.assignment:
            payment += self.origin_tensor[assign[0]][assign[1]]
        print(f"Current payment: {payment}")
        print(f"assignment: {self.assignment}")
    def dfs(self,row, used_cols, path, zero_positions):
        """
        深さ優先探索で独立な0の組み合わせを見つける
        引数:
            row       : 今割り当てようとしている行インデックス
            used_cols : これまでに選んだ列（訪問済みノードの集合）
            path      : これまでに選んだ (row, col) のリスト
        """
        if row == self.N:
            return path
        for col in zero_positions[row]:
            if col in used_cols:
                continue
            new_used_cols = used_cols | {col}
            new_path = path + [(row, col)]

            res = self.dfs(row + 1, new_used_cols, new_path, zero_positions)
            if res is not None:
                return res 
        return None
    def zero_count(self, now_tensor):
        tmp_tensor = now_tensor
        num_zero = []
        for i in range(self.N):
            row = tmp_tensor[i]
            zero_count = int(np.sum(row == 0))
            num_zero.append(("row", i, zero_count))
        for j in range(self.N):
            col = tmp_tensor[:, j]
            zero_count = int(np.sum(col == 0))
            num_zero.append(("col", j, zero_count))
        num_zero.sort(key=lambda x: x[2], reverse=True)
        return num_zero

    def step1(self,now_tensor):
        """
        Step 1:すべての列とすべての行について、それぞれの内部にある最小値を引く
        """
        tmp_tesor = now_tensor
        for i in range(self.N):
            min_value = min(tmp_tesor[i])
            for j in range(self.N):
                tmp_tesor[i][j] -= min_value
        for j in range(self.N):
            col_values = [tmp_tesor[i][j] for i in range(self.N)]
            min_value = min(col_values)
            for i in range(self.N):
                tmp_tesor[i][j] -= min_value
        return tmp_tesor
    def step2(self, now_tensor):
        """
        Step 2:0をカバーする最小の行と列の組み合わせを見つける
        """
        zero_positions = [[j for j in range(self.N) if now_tensor[i, j] == 0] for i in range(self.N)]
        assignment = self.dfs(0, set(), [], zero_positions)
        return assignment
    def step3(self,now_tensor):
        tmp_tensor = now_tensor.copy()
        markded_tensor = now_tensor.copy()
        print("before step3 processing")
        print(f"{tmp_tensor}")
        while (markded_tensor == 0).any():
            num_zero = self.zero_count(tmp_tensor)
            zero_fill = None
            for data in num_zero:
                if data[2] != self.N:
                    zero_fill = data
                    break
            if zero_fill[0] == "row":
                print(f"row {zero_fill[1]} selected")
                markded_tensor[zero_fill[1], :] = INF_
                tmp_tensor[zero_fill[1], :] = 0
            else:
                print(f"col {zero_fill[1]} selected")
                markded_tensor[:, zero_fill[1]] = INF_
                tmp_tensor[:, zero_fill[1]] = 0
        return markded_tensor
    def step4(self,now_tensor, marked_tensor):
        tmp_tensor = now_tensor
        min_value = INF_
        for i in range(self.N):
            for j in range(self.N):
                if marked_tensor[i][j] != INF_:
                    if min_value > tmp_tensor[i][j]:
                        min_value = tmp_tensor[i][j]
        for i in range(self.N):
            for j in range(self.N):
                if marked_tensor[i][j] == INF_ :
                    if tmp_tensor[i][j] != 0:tmp_tensor[i][j] += min_value
                else:
                    tmp_tensor[i][j] -= min_value
        return tmp_tensor
cost = np.array(
    [
        [7, 8, 5, 6],
        [6, 5, 9, 10],
        [4, 1, 10, 7],
        [3, 4, 6, 5],
    ]
)
hungarian = Hungarian(cost)

before step3 processing
[[2 3 0 0]
 [1 0 4 4]
 [3 0 9 5]
 [0 1 3 1]]
row 0 selected
col 1 selected
row 3 selected
After step3 processing, the marked tensor is:
[[1000000000 1000000000 1000000000 1000000000]
 [         1 1000000000          4          4]
 [         3 1000000000          9          5]
 [1000000000 1000000000 1000000000 1000000000]]
[[3 4 0 0]
 [0 0 3 3]
 [2 0 8 4]
 [0 2 4 2]]
before step3 processing
[[3 4 0 0]
 [0 0 3 3]
 [2 0 8 4]
 [0 2 4 2]]
row 0 selected
col 0 selected
col 1 selected
After step3 processing, the marked tensor is:
[[1000000000 1000000000 1000000000 1000000000]
 [1000000000 1000000000          3          3]
 [1000000000 1000000000          8          4]
 [1000000000 1000000000          4          2]]
[[5 6 0 0]
 [0 0 1 1]
 [4 0 6 2]
 [0 4 2 0]]
Current payment: 17
assignment: [(0, 2), (1, 0), (2, 1), (3, 3)]


In [38]:
def find_assignment_zero(cost):
    """
    cost: 0/非0 の行列 (numpy array でもリストでもOK)
    返り値: 割当 (row, col) のリスト or None
    """
    n_rows, n_cols = cost.shape

    # 各行にどの列に 0 があるかを前計算しておく
    zero_positions = [
        [j for j in range(n_cols) if cost[i, j] == 0] for i in range(n_rows)
    ]

    def dfs(row, used_cols, path):
        """
        row       : 今割り当てようとしている行インデックス
        used_cols : これまでに選んだ列（訪問済みノードの集合）
        path      : これまでに選んだ (row, col) のリスト
        """
        # すべての行に対して 1 つずつ 0 が選べたら成功
        if row == n_rows:
            return path

        # この行にある 0 を順番に試す
        for col in zero_positions[row]:
            if col in used_cols:
                # すでに他の行でこの列を使っている → スキップ
                continue

            # 訪問済み列（ノード）の集合を更新
            new_used_cols = used_cols | {col}
            new_path = path + [(row, col)]

            res = dfs(row + 1, new_used_cols, new_path)
            if res is not None:
                return res  # 一つでも成功したらそれを返す

        # どの列を選んでも行き詰まった場合
        return None

    return dfs(0, set(), [])


# 動作テスト用の簡単な例
cost = np.array(
    [
        [0, 1, 2],
        [1, 0, 3],
        [4, 0, 0],
    ]
)

assignment = find_assignment_zero(cost)
print(assignment)  # 例: [(0, 0), (1, 1), (2, 2)] など

[(0, 0), (1, 1), (2, 2)]
